In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics         import classification_report, roc_auc_score, f1_score, recall_score
from utils.preprocessing     import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils        import get_model_train_eval
from utils.feature_engineering import add_statistical_features, drop_highly_correlated_features


In [2]:
# 데이터 로딩 및 기본 전처리
train, test = load_data()
X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=['ID'], axis=1)

In [ ]:
# zero_count_rate 제거했을때 제거될 컬럼수 149개 잔존
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99)


In [4]:
print(X_features.shape)

# var3 처리
X_features['var3'] = X_features['var3'].replace(-999999, 2)

(76020, 149)


In [ ]:
# # add_statistical_features() before corr
# X_features = add_statistical_features(X_features)
# X_test     = add_statistical_features(X_features)

In [5]:
# 상관계수 높은 feature들 삭제하기 default 0.95
X_reduced, to_drop = drop_highly_correlated_features(X_features)
X_test_reduced = X_test.drop(to_drop, axis=1)

In [6]:
print(X_reduced.shape, X_test_reduced.shape)

(76020, 96) (75818, 96)


In [7]:
# 삭제된 컬럼 개수 확인
print("Train에서 삭제된 컬럼 개수:", len(to_drop))

# for i, col in enumerate(sorted(to_drop), start=1):
#     print(f"{i:>2} : {col}")

Train에서 삭제된 컬럼 개수: 53


In [48]:
# 리스트를 Pandas Series로 변환
series = pd.Series(sorted(to_drop), name="Dropped_Columns")

# CSV 파일로 저장
series.to_csv("../data/99perCorr95DroppedColumns_20251122.xls", index=False)


In [ ]:
# 스케일링 
# X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)

In [15]:
# 학습/테스트 데이터 분리
# X_train, X_val, y_train, y_val = data_split(X_train_scaled, y_labels)
X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)

In [16]:
model_name = 'RandomForest_99per_corrTh95_NoTScaled_ne390_maxDepth25_classWeight110_msl1_mss7' 
# model_name = 'RandomForest_99per_corrTh95_NoTScaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
# model_name = 'RandomForest_99per_corrTh95_Scaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
# Best Option 적용
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 390,
  max_depth    = 25, 
  class_weight = {0:1, 1:10}, # 클래스별 가중치
  min_samples_leaf  = 1, 
  min_samples_split = 7,
  n_jobs            = -1 # 병렬처리 여부   
  
)

# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)
result_text = '''
✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95_Scaled_ne390_maxDepth25_classWeight12_msl1_mss7.pkl
  파일 크기: 79.76 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8409, 정확도: 0.9602, 정밀도: 0.3636, 재현율: 0.0066, F1: 0.0131
오차행렬:
[[14595     7]
 [  598     4]]
실행 시간: 11.42557954788208
'''

✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95_NoTScaled_ne390_maxDepth25_classWeight110_msl1_mss7.pkl
  파일 크기: 83.41 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8263, 정확도: 0.9133, 정밀도: 0.1997, 재현율: 0.3953, F1: 0.2653
오차행렬:
[[13648   954]
 [  364   238]]
실행 시간: 9.120373964309692


In [ ]:
#threshold  dropped_features  AUC       F1        Recall
# 0.95      53                0.842169  0.016287  0.008306 (Not Scaled)
# 0.95      53                0.8409    0.0131    0.0066   (Scaled)
# 0.95      57                0.8359    0.0350    0.0183   (Not Scaled)
# 0.95      53                0.8318    0.0218    0.0113   (KFold)

In [13]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_recall_curve, auc

def stratified_cv_randomforest(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    auc_scores, f1_scores, recall_scores, pr_auc_scores = [], [], [], []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # RandomForest 모델 (가중치 조정 가능)
        rf_clf = RandomForestClassifier(
            random_state=0,
            n_estimators=390,
            max_depth=25,
            class_weight={0:1, 1:10},  # 필요시 {0:1, 1:2} 등으로 변경
            min_samples_leaf=1,
            min_samples_split=7,
            n_jobs=-1
        )
        
        rf_clf.fit(X_train, y_train)
        
        y_pred_proba = rf_clf.predict_proba(X_val)[:, 1]
        y_pred = rf_clf.predict(X_val)
        
        # ROC-AUC
        auc_score = roc_auc_score(y_val, y_pred_proba)
        # F1
        f1 = f1_score(y_val, y_pred)
        # Recall
        recall = recall_score(y_val, y_pred)
        # PR-AUC
        precision, recall_curve, _ = precision_recall_curve(y_val, y_pred_proba)
        pr_auc = auc(recall_curve, precision)
        
        auc_scores.append(auc_score)
        f1_scores.append(f1)
        recall_scores.append(recall)
        pr_auc_scores.append(pr_auc)
        
        print(f"[Fold {fold}] ROC-AUC={auc_score:.4f}, PR-AUC={pr_auc:.4f}, F1={f1:.4f}, Recall={recall:.4f}")
    
    print("\n=== Stratified K-Fold 평균 성능 ===")
    print(f"ROC-AUC 평균: {np.mean(auc_scores):.4f}")
    print(f"PR-AUC 평균:  {np.mean(pr_auc_scores):.4f}")
    print(f"F1 평균:      {np.mean(f1_scores):.4f}")
    print(f"Recall 평균:  {np.mean(recall_scores):.4f}")
    
    return None #auc_scores, pr_auc_scores, f1_scores, recall_scores

In [ ]:
# Kfold 적용 : class_weight=1:2, 1:20적용
stratified_cv_randomforest(X_reduced, y_labels)
result_cw12='''
[Fold 1] AUC=0.8346, F1=0.0260, Recall=0.0133
[Fold 2] AUC=0.8430, F1=0.0066, Recall=0.0033
[Fold 3] AUC=0.8247, F1=0.0317, Recall=0.0166
[Fold 4] AUC=0.8321, F1=0.0161, Recall=0.0083
[Fold 5] AUC=0.8246, F1=0.0289, Recall=0.0150

=== Stratified K-Fold 평균 성능 ===
AUC 평균: 0.8318
F1 평균: 0.0218
Recall 평균: 0.0113
'''

In [ ]:
# Kfold 적용 : class_weight=1:10적용
stratified_cv_randomforest(X_reduced, y_labels)
result_cw110 = '''
[Fold 1] ROC-AUC=0.8147, PR-AUC=0.1600, F1=0.2645, Recall=0.3977
[Fold 2] ROC-AUC=0.8239, PR-AUC=0.1647, F1=0.2603, Recall=0.3927
[Fold 3] ROC-AUC=0.8170, PR-AUC=0.1548, F1=0.2510, Recall=0.3671
[Fold 4] ROC-AUC=0.8054, PR-AUC=0.1587, F1=0.2755, Recall=0.4020
[Fold 5] ROC-AUC=0.8239, PR-AUC=0.1659, F1=0.2506, Recall=0.3704

=== Stratified K-Fold 평균 성능 ===
ROC-AUC 평균: 0.8170
PR-AUC 평균:  0.1608
F1 평균:      0.2604
Recall 평균:  0.3860
'''

[Fold 1] ROC-AUC=0.8147, PR-AUC=0.1600, F1=0.2645, Recall=0.3977
[Fold 2] ROC-AUC=0.8239, PR-AUC=0.1647, F1=0.2603, Recall=0.3927
[Fold 3] ROC-AUC=0.8170, PR-AUC=0.1548, F1=0.2510, Recall=0.3671
[Fold 4] ROC-AUC=0.8054, PR-AUC=0.1587, F1=0.2755, Recall=0.4020
[Fold 5] ROC-AUC=0.8239, PR-AUC=0.1659, F1=0.2506, Recall=0.3704

=== Stratified K-Fold 평균 성능 ===
ROC-AUC 평균: 0.8170
PR-AUC 평균:  0.1608
F1 평균:      0.2604
Recall 평균:  0.3860
